In [1]:
from dotenv import load_dotenv

load_dotenv('E:/Courses/Intro to Langchain/lca-lc-foundations-main/example.env')

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is seeking information about Lunapolis, including its capital, weather, population of cheese miners, and union activities.\n\n## SUMMARY\n- The capital of the moon is Lunapolis.\n- Lunapolis has clear skies with a temperature range of 120°C to -100°C.\n- There are 100,000 cheese miners living in Lunapolis.\n- The cheese miners' union is likely to strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='256bfe47-842c-4769-b4be-f019bfc2d051'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='c1b3b9ae-f4d8-4182-8299-89e5c7666470'),
              AIMessage(content='If I were Lunapolis’ new president, I’d respond with a principled, transp

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is seeking information about Lunapolis, including its capital, weather, population of cheese miners, and union activities.

## SUMMARY
- The capital of the moon is Lunapolis.
- Lunapolis has clear skies with a temperature range of 120°C to -100°C.
- There are 100,000 cheese miners living in Lunapolis.
- The cheese miners' union is likely to strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='89377589-0166-4992-a3a1-85a3864d5dfe'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='5619586f-8b6a-405a-8fc9-17e97e3fe624', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='8eae0445-5c17-4776-9100-06b02730caa7'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='981f9d27-0585-4392-a3ee-0cc037a34f94', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='7ecb4f9d-651c-49c2-954a-6289838b9ad2'),
              AIMessage(content='I can’t read the device’s temperature from here. Since it won’

In [8]:
print(response["messages"][-1].content)

I can’t read the device’s temperature from here. Since it won’t turn on, you can’t check internal temps with software yet. Here’s what you can do right now:

- Check the exterior: Carefully feel the device from a short distance. If it’s uncomfortably hot (you can’t keep your hand there for more than a second or two), power it off and unplug it. Let it cool on a non-flammable surface.

- Power and connections: Make sure you’re using the original charger, try a different outlet, and disconnect any peripherals.

- Hard reset basics (if applicable): 
  - Laptop: with the charger connected, press and hold the power button for 15–20 seconds, then try turning it on again.
  - Desktop/other devices: try a power cycle by unplugging for a minute, then plugging back in and pressing the power button.

- If it still won’t turn on after cooling and power-cycling, there may be a hardware issue (battery, power supply, motherboard, etc.). Consider service or a diagnostic check.

If you can tell me the 